# Feature-Based Baseline Model: Logistic Regression

In this notebook, I build a classical feature-based baseline for named entity recognition using logistic regression. The idea is simple, each token is classified independently, but I still provide the model with a small amount of local context through hand crafted features.

This baseline is important for the project because it gives me a strong non-neural comparison point before I move to the BiLSTM model. It also helps me test whether visible cues such as capital letters, suffixes, POS tags, and neighbouring words are already enough to capture a large part of the NER task.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### **1. Load Dataset**

The dataset stores one token per line with four columns: 'word', 'POS', 'chunk', and 'NER tag'. Blank lines separate sentences. I load each sentence as a list of token dictionaries so I can use richer features later. Also, I keep the dataset in a different format compared to the 'data exploration notebook' because I need POS and chunk information as input features for the feature-based model.

In [5]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [6]:
from src.load_dataset import load_dataset_lr

train_sentences = load_dataset_lr("../data/raw/train.txt")
valid_sentences = load_dataset_lr("../data/raw/valid.txt")
test_sentences = load_dataset_lr("../data/raw/test.txt")

print(f"Train sentences: {len(train_sentences): }")
print(f"Valid sentences: {len(valid_sentences): }")
print(f"Test sentences:  {len(test_sentences): }")

Train sentences:  14041
Valid sentences:  3250
Test sentences:   3453


In [7]:
train_sentences[0]

[{'word': 'EU', 'pos': 'NNP', 'chunk': 'B-NP', 'ner': 'B-ORG'},
 {'word': 'rejects', 'pos': 'VBZ', 'chunk': 'B-VP', 'ner': 'O'},
 {'word': 'German', 'pos': 'JJ', 'chunk': 'B-NP', 'ner': 'B-MISC'},
 {'word': 'call', 'pos': 'NN', 'chunk': 'I-NP', 'ner': 'O'},
 {'word': 'to', 'pos': 'TO', 'chunk': 'B-VP', 'ner': 'O'},
 {'word': 'boycott', 'pos': 'VB', 'chunk': 'I-VP', 'ner': 'O'},
 {'word': 'British', 'pos': 'JJ', 'chunk': 'B-NP', 'ner': 'B-MISC'},
 {'word': 'lamb', 'pos': 'NN', 'chunk': 'I-NP', 'ner': 'O'},
 {'word': '.', 'pos': '.', 'chunk': 'O', 'ner': 'O'}]

In [8]:
def count_tokens(sentences):
    return sum(len(sent) for sent in sentences)

summary_df = pd.DataFrame({
    'split': ['train', 'valid', 'test'],
    'sentences': [len(train_sentences), len(valid_sentences), len(test_sentences)],
    'tokens': [count_tokens(train_sentences), count_tokens(valid_sentences), count_tokens(test_sentences)],
})
summary_df

,split,sentences,tokens
0,train,14041,203621
1,valid,3250,51362
2,test,3453,46435


### **2. Define Hand Crafted Features**

I manually describe each token with a feature dictionary.

- the current word and lowercase form
- prefixes and suffixes
- capitalisation and digit patterns
- POS and chunk tags
- one-token context on the left and right
- sentence boundary markers

It will give logistic regression enough information to learn many common NER patterns.

In [9]:
from src.hand_crafted_features import token_to_features

example_token = token_to_features(train_sentences[0], 1)
example_token

{'word.lower': 'rejects',
 'word[-3:]': 'cts',
 'word[-2:]': 'ts',
 'word[:3]': 'rej',
 'word[:2]': 're',
 'word.isupper': False,
 'word.istitle': False,
 'word.isdigit': False,
 'word.has_hyphen': False,
 'word.shape': 'xxxxxxx',
 'pos': 'VBZ',
 'pos[:2]': 'VB',
 'chunk': 'B-VP',
 '-1:word.lower': 'eu',
 '-1:word.istitle': False,
 '-1:word.isupper': True,
 '-1:pos': 'NNP',
 '-1:chunk': 'B-NP',
 '+1:word.lower': 'german',
 '+1:word.istitle': True,
 '+1:word.isupper': False,
 '+1:pos': 'JJ',
 '+1:chunk': 'B-NP'}

I add some extra features differently from surface cues in data exploration.

First of all, I extract word shape features to capture general patterns such as capitalisation and digits to help the model generalise to unseen words, since tokens with similar shapes often belong to the same entity type. 

Secondly, I use the first two characters of the POS tag to create a more general feature to reduce 'sparsity' and allows the model to learn broader patterns, such as distinguishing nouns from verbs.

Moreover, including features from the previous and next tokens to provide local context information to the model disambiguate entities, since the meaning of a word often depends on surrounding words. Only a small set of features is used for capturing important signals while keeping the model simple.

### **3. Vectorise the Features**

Scikit-learn needs a 2D matrix, so I flatten all sentence-level features into token-level examples. Then I convert the feature dictionaries into a sparse matrix with 'DictVectorizer' and encode the labels with 'LabelEncoder'.

In [11]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Convert a sentence to list of feature dictionaries
def sentence_to_features(sentence):
    return [token_to_features(sentence, i) for i in range(len(sentence))]

# Extract the NER labels
def sentence_to_labels(sentence):
    return [token['ner'] for token in sentence]

# Extract only the word tokens
def sentence_to_tokens(sentence):
    return [token['word'] for token in sentence]

In [19]:
X_train_feat = [feature for sentence in train_sentences for feature in sentence_to_features(sentence)]
X_valid_feat = [feature for sentence in valid_sentences for feature in sentence_to_features(sentence)]
X_test_feat = [feature for sentence in test_sentences for feature in sentence_to_features(sentence)]

y_train_labels = [label for sentence in train_sentences for label in sentence_to_labels(sentence)]
y_valid_labels = [label for sentence in valid_sentences for label in sentence_to_labels(sentence)]
y_test_labels = [label for sentence in test_sentences for label in sentence_to_labels(sentence)]

token_sequences_valid = [sentence_to_tokens(sentence) for sentence in valid_sentences]
token_sequences_test = [sentence_to_tokens(sentence) for sentence in test_sentences]
tag_sequences_valid = [sentence_to_labels(sentence) for sentence in valid_sentences]
tag_sequences_test = [sentence_to_labels(sentence) for sentence in test_sentences]

In [23]:
X_train_feat[:2]

[{'word.lower': 'eu',
  'word[-3:]': 'eu',
  'word[-2:]': 'eu',
  'word[:3]': 'eu',
  'word[:2]': 'eu',
  'word.isupper': True,
  'word.istitle': False,
  'word.isdigit': False,
  'word.has_hyphen': False,
  'word.shape': 'XX',
  'pos': 'NNP',
  'pos[:2]': 'NN',
  'chunk': 'B-NP',
  'BOS': True,
  '+1:word.lower': 'rejects',
  '+1:word.istitle': False,
  '+1:word.isupper': False,
  '+1:pos': 'VBZ',
  '+1:chunk': 'B-VP'},
 {'word.lower': 'rejects',
  'word[-3:]': 'cts',
  'word[-2:]': 'ts',
  'word[:3]': 'rej',
  'word[:2]': 're',
  'word.isupper': False,
  'word.istitle': False,
  'word.isdigit': False,
  'word.has_hyphen': False,
  'word.shape': 'xxxxxxx',
  'pos': 'VBZ',
  'pos[:2]': 'VB',
  'chunk': 'B-VP',
  '-1:word.lower': 'eu',
  '-1:word.istitle': False,
  '-1:word.isupper': True,
  '-1:pos': 'NNP',
  '-1:chunk': 'B-NP',
  '+1:word.lower': 'german',
  '+1:word.istitle': True,
  '+1:word.isupper': False,
  '+1:pos': 'JJ',
  '+1:chunk': 'B-NP'}]

In [24]:
y_train_labels[:10]

['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O', 'B-PER']

In [25]:
token_sequences_valid[:5]

[['CRICKET',
  '-',
  'LEICESTERSHIRE',
  'TAKE',
  'OVER',
  'AT',
  'TOP',
  'AFTER',
  'INNINGS',
  'VICTORY',
  '.'],
 ['LONDON', '1996-08-30'],
 ['West',
  'Indian',
  'all-rounder',
  'Phil',
  'Simmons',
  'took',
  'four',
  'for',
  '38',
  'on',
  'Friday',
  'as',
  'Leicestershire',
  'beat',
  'Somerset',
  'by',
  'an',
  'innings',
  'and',
  '39',
  'runs',
  'in',
  'two',
  'days',
  'to',
  'take',
  'over',
  'at',
  'the',
  'head',
  'of',
  'the',
  'county',
  'championship',
  '.'],
 ['Their',
  'stay',
  'on',
  'top',
  ',',
  'though',
  ',',
  'may',
  'be',
  'short-lived',
  'as',
  'title',
  'rivals',
  'Essex',
  ',',
  'Derbyshire',
  'and',
  'Surrey',
  'all',
  'closed',
  'in',
  'on',
  'victory',
  'while',
  'Kent',
  'made',
  'up',
  'for',
  'lost',
  'time',
  'in',
  'their',
  'rain-affected',
  'match',
  'against',
  'Nottinghamshire',
  '.'],
 ['After',
  'bowling',
  'Somerset',
  'out',
  'for',
  '83',
  'on',
  'the',
  'opening',


I flatten the feature representations and labels across all sentences. This results in a single list of feature dictionaries (X_train_feat) and a corresponding list of labels (y_train_labels), where each element represents one token. This format is required for training the classifier using standard machine learning libraries.

In addition, I keep the original token and label sequences for the validation and test sets. These are used later for evaluation at the sentence level, which is necessary for computing sequence-based metrics and analysing model predictions.

In [ ]:
# Convert feature dictionaries to sparse vectors
vectorizer = DictVectorizer(sparse=True)
X_train = vectorizer.fit_transform(X_train_feat)
X_valid = vectorizer.transform(X_valid_feat)
X_test = vectorizer.transform(X_test_feat)

# Encode string NER labels into integers
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_labels)
y_valid = label_encoder.transform(y_valid_labels)
y_test = label_encoder.transform(y_test_labels)

print(f"Training matrix shape: {X_train.shape}")
print(f"Validation matrix shape: {X_valid.shape}")
print(f"Test matrix shape: {X_test.shape}")
print(f"Number of labels: {len(label_encoder.classes_)}")

Training matrix shape: (203621, 72781)
Validation matrix shape: (51362, 72781)
Test matrix shape: (46435, 72781)
Number of labels: 9


I convert the feature dictionaries into numerical vectors using a 'DictVectorizer', since machine learning models require numerical input. This step transforms each token’s feature dictionary into a high-dimensional sparse vector representation. The vectorizer is fitted on the training data and then applied to the validation and test sets to ensure consistency.

I also encode the NER labels into numerical form using a 'LabelEncoder', as the model cannot work with string labels directly. The encoder is fitted on the training labels and reused for validation and test labels to maintain the same mapping.